# 01 · 完整预处理与 HDF5 导出

这是预处理的唯一主 Notebook。顺序固定为：输入/事件检查 → 电刺激定位表检查 → 信号与定位表交集 → 全长滤波 → 坏道决定 → 邻接双极参考 → epoch → HDF5 → 验证 → 读取 Task 2 子条件 → 均值/阴影可视化。

当前记忆颜色解码的第一阶段只选 Task 2 的 `gray` 条件；`true` 和 `false` 已保存在同一个 HDF5 中，后续再按需读取。

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r'E:/liulab_project/Project_colorieeg_2026')
MODULE_ROOT = PROJECT_ROOT / 'color_analyse_0727'
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

from pipeline.condition_registry import conditions_for_group
from pipeline.config import PROCESS_DATA_ROOT, QC_ROOT, METADATA_ROOT
from pipeline.hdf5_io import inspect_hdf5, load_condition_epochs
from pipeline.epoch_plots import plot_hdf5_conditions

RUN_REBUILD = False
RUN_VALIDATE = False
RUN_FILTER_DEMO = False
RUN_PLOT = False

## 1. 条件注册表与事件库存

In [ ]:
condition_registry = pd.read_csv(QC_ROOT / 'condition_registry.csv')
event_inventory = pd.read_csv(QC_ROOT / 'event_inventory.csv')
display(condition_registry)
display(event_inventory.head(12))

## 2. 电刺激行为学记录与定位表

定位表已由 `metadata/stimulation_behavioral_annotation.csv` 更新：atlas 列不改；最后一列 `color_with_sti` 只对记录中有清晰颜色证据的接触点置为 `True`，弱颜色证据单独保留为 review。原始定位表备份在 `metadata/localization_original/`。

In [ ]:
stim_pairs = pd.read_csv(METADATA_ROOT / 'stimulation_behavioral_annotation.csv')
stim_contacts = pd.read_csv(METADATA_ROOT / 'stimulation_behavioral_electrodes.csv')
display(stim_pairs.groupby(['subject', 'response_class']).size().unstack(fill_value=0))
display(stim_contacts.query('color_with_sti == True')[['subject', 'channel', 'stim_color_evidence', 'stim_behavior_pairs']])

In [ ]:
manifest = pd.read_csv(METADATA_ROOT / 'electrode_manifest_by_task.csv')
manifest_summary = (manifest.groupby(['subject', 'task_num'])
                    .agg(signal_channels=('signal_present', 'sum'),
                         localized_channels=('localized', 'sum'),
                         analysis_centers=('analysis_center_eligible', 'sum'))
                    .reset_index())
manifest_summary

## 3. 坏道决定

QC 只使用最终滤波后的全长信号。候选表不会自动删除通道；只有 `manual_channel_decisions.csv` 或 QC 主表中明确写成 `exclude` 的通道才会在导出时排除。当前已记录的人工排除包括 test003 的 B15/I1/I2/I3、test004 K3、test005 G1、test007 18。

In [ ]:
manual_decisions = pd.read_csv(METADATA_ROOT / 'manual_channel_decisions.csv')
qc_decisions = pd.read_csv(QC_ROOT / 'bad_channel_candidates.csv')
display(manual_decisions)
display(qc_decisions.query("manual_decision.str.lower() == 'exclude'", engine='python')[['subject', 'channel', 'manual_decision', 'comment']])

## 4. 全长信号处理

`filter_continuous` 接收整个连续通道，不是在可视化窗口上滤波。顺序是：按通道中位数去 DC → 1000 Hz 重采样到 500 Hz → 1–200 Hz 零相位带通 → 50/100/150 Hz 零相位陷波。滤波后再做 QC；不会做 event-wise baseline correction。

In [ ]:
if RUN_FILTER_DEMO:
    from pipeline.io_seeg import load_set_metadata, open_fdt
    from pipeline.signal_processing import filter_continuous
    meta = load_set_metadata('test001', 1)
    raw_full_length = np.asarray(open_fdt(meta)[0:1, :], dtype=np.float64)
    filtered_full_length = filter_continuous(raw_full_length, meta.sfreq)
    print({'raw_shape': raw_full_length.shape, 'filtered_shape': filtered_full_length.shape, 'sfreq_in': meta.sfreq})
else:
    print('Full-length filter demo disabled; exporter uses the same function.')

## 5. 重建 HDF5（可选）

导出使用信号通道与定位表通道的交集，生成邻接双极中心；Task 2 保留 gray/true/false 全部 12 个条件。

In [ ]:
if RUN_REBUILD:
    subprocess.run([sys.executable, str(MODULE_ROOT / 'scripts' / 'build_electrode_manifest.py')], check=True)
    subprocess.run([sys.executable, str(MODULE_ROOT / 'scripts' / 'build_hdf5.py')], check=True)
else:
    print('HDF5 rebuild disabled; existing process_data outputs will be inspected.')

## 6. HDF5 验证

In [ ]:
if RUN_VALIDATE:
    subprocess.run([sys.executable, str(MODULE_ROOT / 'scripts' / 'validate_hdf5.py')], check=True)
validation = pd.read_csv(METADATA_ROOT / 'hdf5_validation_report.csv')
validation[['subject', 'task_num', 'n_channels', 'n_times', 'conditions', 'status']]

In [ ]:
sample_h5 = PROCESS_DATA_ROOT / 'test001' / 'task2_epoched_1_200Hz.h5'
sample_info = inspect_hdf5(sample_h5)
{key: sample_info[key] for key in ['subject', 'task_num', 'n_channels', 'n_times', 'time_start_ms', 'time_end_ms', 'condition_names', 'trial_counts']}

## 7. Task 2 条件选择模板

读取时只选择分析子集，不改写 HDF5。第一阶段使用 gray；后续 true/false 直接切换 `group`。

In [ ]:
gray_conditions = conditions_for_group(2, 'gray')
true_conditions = conditions_for_group(2, 'true')
false_conditions = conditions_for_group(2, 'false')
print('gray:', gray_conditions)
print('true:', true_conditions)
print('false:', false_conditions)
gray_epoch_example = load_condition_epochs(sample_h5, gray_conditions[0])
print('example shape (trials, bipolar_centers, time):', gray_epoch_example.shape)

## 8. Epoch 均值与阴影示例图

`plot_hdf5_conditions` 是可复用工具：每条线是跨 trial 均值，阴影默认为 SEM，也可以改为 `shade='sd'`。本次示例选取 test001 的 A2、D3、G3、G6，并分别画 Task 1 的 face color/gray 与 Task 2 的四种灰色水果。

In [ ]:
example_channels = ['A2', 'D3', 'G3', 'G6']
example_output = MODULE_ROOT / 'result' / 'epoch_examples'
example_output.mkdir(parents=True, exist_ok=True)

if RUN_PLOT:
    import matplotlib.pyplot as plt
    fig, axes = plot_hdf5_conditions(
        PROCESS_DATA_ROOT / 'test001' / 'task1_epoched_1_200Hz.h5',
        ['face_color', 'face_gray'],
        example_channels,
        shade='sem',
        title='test001 Task 1: face color vs gray',
    )
    fig.savefig(example_output / 'test001_task1_face_color_vs_gray.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('Plotting disabled. Existing MATLAB bridge figures are in result/epoch_examples.')

In [ ]:
if RUN_PLOT:
    import matplotlib.pyplot as plt
    fig, axes = plot_hdf5_conditions(
        PROCESS_DATA_ROOT / 'test001' / 'task2_epoched_1_200Hz.h5',
        list(gray_conditions),
        example_channels,
        shade='sem',
        title='test001 Task 2: gray fruit conditions',
    )
    fig.savefig(example_output / 'test001_task2_gray_fruits.png', dpi=160, bbox_inches='tight')
    plt.show()